#### Инициализация Keras

In [1]:
import os

os.environ["KERAS_BACKEND"] = "torch"
import keras
import torch

print(keras.__version__)

device = "cpu"
keras.utils.set_random_seed(42)

if torch.backends.mps.is_available():
    device = "mps"
    print("Ускорение GPU Apple Metal активно")
elif torch.cuda.is_available():
    device = "cuda"
    print("Ускорение GPU NVIDIA CUDA активно")
else:
    try:
        import torch_directml

        device = torch_directml.device()
        print("Ускорение GPU AMD DirectML активно")
    except ImportError:
        print("Используется CPU")


3.14.0
Ускорение GPU Apple Metal активно


#### Загрузка набора данных для задачи классификации

База данных MNIST (сокращение от "Modified National Institute of Standards and Technology") — объёмная база данных образцов рукописного написания цифр. База данных является стандартом, предложенным Национальным институтом стандартов и технологий США с целью обучения и сопоставления методов распознавания изображений с помощью машинного обучения в первую очередь на основе нейронных сетей. Данные состоят из заранее подготовленных примеров изображений, на основе которых проводится обучение и тестирование систем.

База данных MNIST содержит 60000 изображений для обучения и 10000 изображений для тестирования.

In [2]:
from keras.datasets import mnist

(X_train_raw, y_train_raw), (X_valid_raw, y_valid_raw) = mnist.load_data()

#### Предобработка данных

Количество классов - 10 (от 0 до 9).

Все изображения из X трансформируются в матрицы 28*28 признака и нормализуются.

Для целевых признаков применяется унитарное кодирование в бинарные векторы длиной 10 (нормализация).

Четвертое измерение в reshape определяет количество цветовых каналов.

Используется только один канал, так как изображения не цветные.

Для цветных изображений следует использовать три канала (RGB).

In [3]:
n_classes = 10

X_train = X_train_raw.reshape(60000, 28, 28, 1).astype("float32") / 255
X_valid = X_valid_raw.reshape(10000, 28, 28, 1).astype("float32") / 255
y_train = keras.utils.to_categorical(y_train_raw, n_classes)
y_valid = keras.utils.to_categorical(y_valid_raw, n_classes)

display(X_train[0])
display(y_train[0])

array([[[0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ]],

       [[0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        

array([0., 0., 0., 0., 0., 1., 0., 0., 0., 0.])

### Архитектура LeNet-5

Изменения относительно оригинальной архитектуры:
- Увеличение фильтров в первом сверточном слое с 6 до 32 и втором сверточном слое с 16 до 64.
- Снижение количества субдискретизации активаций до одной вместо двух, только после второго сверточного слоя.
- Применение функции активации ReLU.

#### Проектирование архитектуры LeNet-5

In [4]:
from keras.layers import Conv2D, Dense, Dropout, Flatten, InputLayer, MaxPooling2D
from keras.models import Sequential

lenet_model = Sequential()

# Входной слой
lenet_model.add(InputLayer(shape=(28, 28, 1)))

# Первый скрытый слой
lenet_model.add(Conv2D(32, kernel_size=(3, 3), activation="relu"))

# Второй скрытый слой
lenet_model.add(Conv2D(64, kernel_size=(3, 3), activation="relu"))
lenet_model.add(MaxPooling2D(pool_size=(2, 2)))
lenet_model.add(Dropout(0.25))

# Третий скрытый слой
lenet_model.add(Flatten())
lenet_model.add(Dense(128, activation="relu"))
lenet_model.add(Dropout(0.5))

# Выходной слой
lenet_model.add(Dense(n_classes, activation="softmax"))

lenet_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 24, 24, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,179,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,199,882 (4.58 MB)

 Trainable params: 1,199,882 (4.58 MB)

 Non-trainable params: 0 (0.00 B)

#### Обучение глубокой модели

In [5]:
lenet_model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

lenet_model.fit(
    X_train,
    y_train,
    batch_size=128,
    epochs=10,
    validation_data=(X_valid, y_valid),
)

Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 18s 36ms/step - accuracy: 0.9281 - loss: 0.2360 - val_accuracy: 0.9816 - val_loss: 0.0568
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 15s 33ms/step - accuracy: 0.9747 - loss: 0.0837 - val_accuracy: 0.9880 - val_loss: 0.0338
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 15s 33ms/step - accuracy: 0.9810 - loss: 0.0627 - val_accuracy: 0.9887 - val_loss: 0.0341
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 15s 33ms/step - accuracy: 0.9839 - loss: 0.0521 - val_accuracy: 0.9896 - val_loss: 0.0302
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 15s 33ms/step - accuracy: 0.9863 - loss: 0.0430 - val_accuracy: 0.9893 - val_loss: 0.0315
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 15s 33ms/step - accuracy: 0.9884 - loss: 0.0365 - val_accuracy: 0.9915 - val_loss: 0.0264
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 19s 40ms/step - accuracy: 0.9896 - loss: 0.0320 - val_accuracy: 0.9916 - val_loss: 0.0266
Epoch 8/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 15s 33ms/step - accuracy: 0.9901 - loss: 0.0310 - 

#### Оценка качества модели

Точность модели на валидационной выборке больше 99 %

In [6]:
lenet_model.evaluate(X_valid, y_valid)

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9923 - loss: 0.0292


[0.02918746881186962, 0.9922999739646912]